<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Multiple Linear Regression

*Session 5 · Notebook 02.03 · Lecture · Student version*

## Overview

Simple linear regression used one feature. **Multiple** linear regression uses several at once: it predicts a continuous target from a weighted combination of many features. This notebook builds the idea step by step. We first show it on a small **synthetic** dataset where we know the true relationship, so you can watch the model recover it, and then apply it to a real business dataset (predicting a startup's profit), including how to handle a categorical feature and how to read several coefficients at once.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how multiple linear regression extends the straight line to many features.
- Encode a categorical feature for a linear model, and avoid the dummy-variable trap.
- Train, predict and evaluate a multiple regression (R-squared, MAE, MSE, RMSE).
- Interpret several coefficients, each as an effect 'holding the other features constant'.
- Recognise multicollinearity and why it complicates interpretation.

## Prerequisites

- Session 5 notebook 02.02 (simple linear regression) and 01.03 (encoding categoricals).
- Comfort with `train_test_split` and the regression metrics.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is multiple linear regression?](#sec2)
3. [See it first on synthetic data](#sec3)
4. [Explore the real dataset](#sec4)
5. [Encode the categorical feature](#sec5)
6. [Split into training and test sets](#sec6)
7. [Train the model](#sec7)
8. [Predict and evaluate](#sec8)
9. [Cross-validation: a more reliable score](#seccv)
10. [Interpret the coefficients](#sec9)
11. [Exercises](#exercises)
12. [Challenge](#challenge)
13. [Key Takeaways](#takeaways)
14. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

We use `50_Startups.csv` (50 companies: their R&D, administration and marketing spend, their US state, and their profit), read from the repo-root `datasets/` folder (two levels up).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# dataset = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/50_Startups.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# dataset = pd.read_csv(session_datasets_http["50_Startups"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# dataset = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/50_Startups.csv", header=True, inferSchema=True).toPandas()
dataset = pd.read_csv('../../datasets/Session_5/50_Startups.csv')
print('shape:', dataset.shape)
dataset.head()

shape: (50, 5)


,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Real outcomes rarely depend on a single driver. A loss, an exposure or a claim size is shaped by many factors at once, and multiple linear regression is the simplest model that combines them.

| Multiple regression lets us... | Risk example |
|---|---|
| Combine several drivers into one prediction | Predict expected loss from balance, income, tenure and product together |
| Read each driver's separate effect | "Each extra 1,000 of balance adds X to expected loss, holding income constant" |
| Include categorical factors | Add region or product type alongside numeric drivers |
| Stay transparent and auditable | Every coefficient is a stated, challengeable effect |

The interpretability is the point: a committee can see exactly how each factor moves the prediction.

<a id="sec2"></a>
# Section 2: What is multiple linear regression?

**Definition:** multiple linear regression predicts the target as a weighted sum of several features plus an intercept:

```
y = b0 + b1*x1 + b2*x2 + ... + bn*xn
```

**Example:** predicting a startup's profit from its R&D, administration and marketing spend simultaneously.

**Analogy:** simple regression fits a line through a 2D scatter; with two features the model fits a flat **plane** through a 3D cloud, and with more features a **hyperplane** we cannot draw but the maths is identical.

**Explanation (this is the key idea):**

- Each coefficient `b_i` is the change in the target for a **one-unit increase in that feature, holding all the other features constant**. That 'holding others constant' is what makes multiple regression more than a set of separate simple regressions.
- The model still fits by **least squares** (minimising the total squared error), exactly as before, just in more dimensions.
- Because coefficients are read 'all else equal', strongly correlated features (**multicollinearity**) make individual coefficients unstable and hard to interpret; we return to this at the end.

**The formula (how the coefficients are found):**

The model is $\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_n x_n$. If we stack the features into a matrix $X$ (with a column of ones for the intercept) and the coefficients into a vector $\boldsymbol{\beta}$, least squares has an exact, direct solution known as the **normal equations**:

$$\boldsymbol{\beta} = (X^\top X)^{-1} X^\top y$$

There is no iteration: every coefficient is computed together in a single linear-algebra step. See the side-note notebook `05_18` for a worked from-scratch check against scikit-learn, and why Lasso cannot be solved this way.

**scikit-learn documentation:** [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

<a id="sec3"></a>
# Section 3: See it first on synthetic data

Before the real, messy dataset, we build a tiny **synthetic** one where we *decide* the true relationship. That lets us check the model does what we expect: recover the coefficients we built in. We create two features and generate the target with known weights (intercept 20, `x1` weight +3.0, `x2` weight -1.5) plus a little random noise.

In [ ]:
rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(50, 10, n)     # e.g. marketing spend
x2 = rng.normal(30, 8, n)      # e.g. number of staff
noise = rng.normal(0, 5, n)    # random variation the model cannot explain

# The TRUE relationship we are building in:
y_true = 20 + 3.0 * x1 - 1.5 * x2 + noise
synth = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y_true})
synth.head()

Now we fit a multiple linear regression on the two features and see whether it recovers the intercept and weights we used. It should land very close to 20, +3.0 and -1.5.

<a id="sec4"></a>
# Section 4: Explore the real dataset

The target is `Profit`. The features are three spending columns (numeric) and `State` (categorical). We inspect the structure and the state breakdown before modelling.

<a id="sec5"></a>
# Section 5: Encode the categorical feature

A linear model needs numbers, so `State` must be encoded. We use one-hot encoding, but with a twist: we **drop one category**.

**Why drop one?** With three states we could make three 0/1 columns, but they always sum to 1, which makes them perfectly redundant with the intercept. That redundancy is the **dummy variable trap** (perfect multicollinearity), and it makes the coefficients unstable. Dropping one state fixes it: the dropped state becomes the **reference**, and each remaining state's coefficient is read *relative to that reference*. `pd.get_dummies(..., drop_first=True)` does this for us.

<a id="sec6"></a>
# Section 6: Split into training and test sets

We separate the features `X` (everything except `Profit`) from the target `y` (`Profit`), then hold out 20% for testing so we can judge the model on data it has not seen.

<a id="sec7"></a>
# Section 7: Train the model

Fitting is one call. `LinearRegression` finds the intercept and the coefficient for every feature at once, by least squares.

<a id="sec8"></a>
# Section 8: Predict and evaluate

We predict the test-set profits and compare them to the truth with the regression metrics: MAE and RMSE are in the target's units (pounds of profit), and R-squared is the share of profit variation the model explains.

<a id="seccv"></a>
# Section 9: Cross-validation, a more reliable score

So far we judged the model on a **single** train/test split. But that score depends on *which* rows happened to land in the test set. With only 50 companies, one split might put the easy cases in the test set (a flattering R-squared) and another the hard ones (a harsh R-squared). A single number can be a lottery.

**Definition:** **k-fold cross-validation** splits the data into `k` equal parts ("folds"). It trains on `k-1` folds and tests on the one held-out fold, then rotates so that **every fold serves as the test set exactly once**. That produces `k` scores, which we average. Every row is used for training and, once, for testing.

**Example:** with `k = 5`, the 50 companies are divided into five groups of ten; the model is trained and scored five times, each time holding out a different group.

**Analogy:** judging a student on a single exam risks catching a bad day. Averaging five exams gives a fairer, more stable picture of their ability. Cross-validation does the same for a model.

**Explanation:**

- You get a **mean** score (the headline estimate) and a **standard deviation** (how much it wobbles between folds). A large spread warns that the estimate is fragile.
- It uses the data efficiently: no single chunk is permanently sacrificed to testing.
- Common choices are `k = 5` or `k = 10`. More folds means more training data per fit, but more computation.
- If you are also *tuning* something (like a penalty strength), you still keep a final untouched test set for the very end; cross-validation is done on the training portion. This is exactly the engine inside `RidgeCV`, `LassoCV` and `GridSearchCV` in the next notebooks.

In [ ]:
# 5-fold cross-validation of ordinary linear regression on the full dataset

> **Note:** the loop below is *repeated random splitting* (the test sets overlap, and a given row can land in several of them or none). That is a different procedure from the disjoint folds of k-fold cross-validation above, where each row is tested exactly once. Here it is used only to show how much a single split's R-squared varies with the luck of the split, which is why it uses more iterations (10) than the 5 folds above.

In [ ]:
# How much does a SINGLE split's R-squared depend on luck? Try 10 different splits.

In [ ]:
# cross_validate reports several metrics at once. Note that scikit-learn uses NEGATIVE error
# scores (it always treats 'higher is better'), so we flip the sign back for a readable RMSE.

The takeaway: a single split gives one noisy estimate, while cross-validation averages several to give a trustworthy one (plus a spread that flags how fragile it is). It is the standard way to estimate performance, and the machinery behind the automatic penalty-tuning (`RidgeCV`, `LassoCV`) in the next two notebooks.

<a id="sec9"></a>
# Section 10: Interpret the coefficients

This is where multiple regression earns its place. Each coefficient is the effect of that feature on profit, **holding the other features constant**. Because the spend features are all in pounds, their coefficients read as 'pounds of profit per pound spent'. The `State` coefficients are the profit difference relative to the dropped reference state.

In [ ]:
# Your turn. Write your solution here:

### Reading the table

**First, a warning about reading raw coefficients.** The table above is sorted by absolute size, so look at what sits on top: `State_Florida` (about -959) and `State_New York` (about 699). Taken at face value that suggests *location* is the biggest driver and R&D Spend (about 0.77) barely matters. That reading is **wrong**, and seeing why is the whole lesson of this section.

Coefficients are expressed in "pounds of profit per **one unit** of that feature", and our features are on wildly different scales:

- `R&D Spend` ranges over roughly 0 to 165,000 pounds. A coefficient of 0.77 means about 0.77 pounds of profit per 1 pound of R&D, which accumulates to an enormous effect across that huge range.
- `State_Florida` is only ever 0 or 1. Its coefficient (about -959) is the *entire* profit difference for being in Florida versus the reference state, and nothing more.

So a large-looking coefficient can belong to an unimportant feature simply because that feature moves over a tiny range (0 to 1), while the most important driver can have a tiny-looking coefficient because it moves over a huge range. **Raw coefficient magnitudes are not comparable across features on different scales.** Each coefficient is still individually valid as a "per unit, all else equal" statement, but you cannot rank importance by eyeballing the column.

The per-feature statements do still hold: each extra pound of R&D adds about 0.77 pounds of profit (all else equal), and being in Florida rather than the reference state is worth about -959 pounds. To *rank* the drivers fairly, though, we first put every feature on the same scale. That is what we do next.

### Comparing importance fairly: standardized coefficients

To compare features on an equal footing, we **standardize** them: subtract each feature's mean and divide by its standard deviation, so every feature has mean 0 and standard deviation 1. After refitting, each coefficient reads as "the change in profit for a **one standard deviation** increase in that feature". Because every feature now moves by the same amount (1 SD), the coefficients become directly comparable and their sizes finally reflect importance. We plot them as a feature-importance chart below.

In [ ]:
# Standardize the features (fit on the training data only), then refit
# Prediction is UNCHANGED by scaling: the test R2 is identical to the original model

In [ ]:
# Feature importance = size of the standardized coefficients

### So why did we not scale the features for the main model?

Because for **ordinary least squares, scaling does not change the predictions at all**: notice the two test R-squared values printed above are identical. OLS just finds different coefficient *numbers* that describe the exact same fitted plane, so the fit, the predictions and R-squared are unchanged. Scaling only earns its keep when:

1. **you want to compare coefficient importance** (what we just did with the chart), or
2. **you use regularization** (Ridge and Lasso, the next two notebooks), where the penalty is applied to the coefficients, so their scale directly affects the result.

For plain prediction with OLS we can therefore leave the features in their natural units and read each coefficient in those units. You will confirm this hands-on in Exercise 4.

<a id="exercises"></a>
# Section 11: Exercises

### Exercise 1: Predict a new startup

Predict the profit of a startup that spends 150,000 on R&D, 120,000 on administration and 300,000 on marketing, based in the reference state (so the state dummy columns are all 0). Build the input with the same columns as `X` and call `regressor.predict`.

In [ ]:
# Your turn. Write your solution here:


### Exercise 2: Which driver matters most?

From the fitted model, print the feature with the largest absolute coefficient. Does it match business intuition about what drives startup profit?

In [ ]:
# Your turn. Write your solution here:


### Exercise 3: Training vs test R-squared

Print the R-squared on the training set and the test set. Are they close (a sign the model generalises rather than overfits)?

In [ ]:
# Your turn. Write your solution here:


### Exercise 4: Does scaling change the predictions?

Standardize the three numeric spend features (`R&D Spend`, `Administration`, `Marketing Spend`) yourself, refit the model, and compare its test R-squared and its first few predictions to the original model. Are the predictions the same? What actually changed?

> *Hint:* fit a `StandardScaler` on the training columns only, then transform both train and test. Or scale by hand with `(col - col.mean()) / col.std()`.

In [ ]:
# Your turn. Write your solution here:


### Exercise 5: Cross-validate with 10 folds

Run a 10-fold cross-validation of `LinearRegression` on `X`, `y` using `cross_val_score` with `scoring='r2'`. Print the mean and standard deviation of the fold scores. Is the spread small (a sign the model performs consistently)?

> *Hint:* use `KFold(n_splits=10, shuffle=True, random_state=0)`.

In [ ]:
# Your turn. Write your solution here:


<a id="challenge"></a>
## Challenge (optional): does dropping weak features hurt?

Multiple regression let us see that R&D dominates. Test whether the weak features are pulling their weight. Train a second model using **only** R&D spend as the feature, compare its test R-squared and RMSE to the full model, and comment on whether the extra features earned their place.

In [ ]:
# Your turn. Write your solution here:


<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Multiple linear regression | Predicts the target from a weighted sum of several features |
| Coefficient meaning | Effect of a feature per unit, **holding the others constant** |
| `pd.get_dummies(..., drop_first=True)` | One-hot encode, dropping one level to avoid the dummy-variable trap |
| `LinearRegression().fit(X, y)` | Learns all coefficients + intercept by least squares |
| `r2_score`, MAE, RMSE | Regression evaluation metrics |
| `.coef_` with feature names | The interpretable table of driver effects |
| Multicollinearity | Correlated features make individual coefficients unstable |


## Conclusion

You can now fit and interpret a regression with many features, handle a categorical driver safely, and read each coefficient as an 'all else equal' effect. You also saw, on synthetic data, that the model genuinely recovers the underlying relationship. Next we let the relationship curve (polynomial regression), then tame large or correlated feature sets with regularisation (Ridge and Lasso).

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Linear Models](https://scikit-learn.org/stable/modules/linear_model.html) LinearRegression with many features.
- [Dummy variable trap](https://en.wikipedia.org/wiki/Dummy_variable_(statistics)) why we drop one category.
- [Multicollinearity](https://en.wikipedia.org/wiki/Multicollinearity) why correlated features complicate interpretation.